# Question 1 — Interactive waste-bin map

**Aditi Saha · HS25M101**

This notebook makes a Folium map of the waste-bin points collected with QField.
Each point has a tooltip, a popup with the collector name, and a photograph
where one was recorded.

The photographs are already in the `web/photos` folder (same file names as in QField).
The popup only points to those files, so they still work after the map is hosted.


In [ ]:
# ------------------------------------------------------------
# 1. IMPORT LIBRARIES
# ------------------------------------------------------------
import geopandas as gpd
import folium
from branca.element import Element


In [ ]:
# ------------------------------------------------------------
# 2. LOAD THE WASTE-BIN GEOPACKAGE
# ------------------------------------------------------------
data_path = r"C:\Users\Ā\Documents\python_class\HS25M101\HS25M101.gpkg"

dustbins = gpd.read_file(
    data_path,
    layer="notes_point"
)

# Photograph names are stored in a separate layer
attachments = gpd.read_file(
    data_path,
    layer="notes_attachments"
)


In [ ]:
# ------------------------------------------------------------
# 3. INSPECT CRS AND CONVERT TO EPSG:4326
# ------------------------------------------------------------
print(dustbins.crs)

dustbins = dustbins.to_crs(epsg=4326)

# Timestamp as text for the tooltip and popup
dustbins["timestamp"] = dustbins["timestamp"].astype(str)

# Fill empty titles and notes
dustbins["title"] = dustbins["title"].fillna("No title")
dustbins["note"] = dustbins["note"].fillna("No note")

# Student who collected the observations
dustbins["collector"] = "Aditi Saha"


In [ ]:
# ------------------------------------------------------------
# 4. JOIN THE PHOTOGRAPH NAMES TO EACH POINT
# ------------------------------------------------------------
# Keep one photograph name for each point uuid
photos = attachments[["note_uuid", "media"]].drop_duplicates("note_uuid")

dustbins = dustbins.merge(
    photos,
    left_on="uuid",
    right_on="note_uuid",
    how="left"
)

# QField stores paths like DCIM/JPEG_....jpg — keep only the file name
dustbins["photo_name"] = dustbins["media"].astype(str).str.split("/").str[-1]
dustbins.loc[dustbins["media"].isna(), "photo_name"] = ""

dustbins[["title", "note", "collector", "photo_name"]].head()


In [ ]:
# ------------------------------------------------------------
# 5. BUILD THE POPUP TEXT FOR EACH POINT
# ------------------------------------------------------------
popups = []

for i, row in dustbins.iterrows():
    html = (
        "<div style='width:240px;font-size:13px;line-height:1.4'>"
        + "<b>" + str(row["title"]) + "</b><br>"
        + "<b>Collected by:</b> " + str(row["collector"]) + "<br>"
        + "<b>Note:</b> " + str(row["note"]) + "<br>"
        + "<b>Recorded at:</b> " + str(row["timestamp"])
    )
    if row["photo_name"] != "":
        html = (
            html
            + "<br><img src='photos/" + row["photo_name"]
            + "' style='margin-top:8px;width:100%;border-radius:4px'>"
        )
    else:
        html = html + "<br><i>No photograph attached</i>"
    html = html + "</div>"
    popups.append(html)

dustbins["popup"] = popups


In [ ]:
# ------------------------------------------------------------
# 6. CALCULATE MAP CENTRE
# ------------------------------------------------------------
center = [
    dustbins.geometry.y.mean(),
    dustbins.geometry.x.mean()
]
print(center)


In [ ]:
# ------------------------------------------------------------
# 7. CREATE THE FOLIUM MAP
# ------------------------------------------------------------
m = folium.Map(
    location=center,
    zoom_start=16,
    tiles=None,
    control_scale=True
)

# Basemap 1 — street map
folium.TileLayer(
    tiles="OpenStreetMap",
    name="Streets"
).add_to(m)

# Basemap 2 — satellite
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri World Imagery",
    name="Satellite"
).add_to(m)


In [ ]:
# ------------------------------------------------------------
# 8. ADD WASTE-BIN POINTS WITH TOOLTIP AND POPUP
# ------------------------------------------------------------
# CircleMarker for each point.
# Tooltip = short info on hover
# Popup = full info + photograph on click

bins = folium.FeatureGroup(name="Waste bins")

for i, row in dustbins.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=7,
        fill=True,
        fill_opacity=0.9,
        weight=1,
        color="white",
        fill_color="#377eb8",
        tooltip=str(row["title"]),
        popup=folium.Popup(row["popup"], max_width=280)
    ).add_to(bins)

bins.add_to(m)


In [ ]:
# ------------------------------------------------------------
# 9. ADD TITLE, LEGEND, NORTH ARROW, LAYER CONTROL AND FIT THE MAP
# ------------------------------------------------------------

# Add a simple north arrow near the top of the study area.

def add_north_arrow(m, lat, lon):
    """Add a simple north arrow to a Folium map."""
    folium.Marker(
        location=[lat, lon],
        icon=folium.DivIcon(
            html=(
                "<div style='font-size:16px; font-weight:bold; "
                "text-align:center; color:black'>"
                "↑<br>N</div>"
            )
        )
    ).add_to(m)


title_html = """
<div style="position:fixed;top:12px;left:50%;transform:translateX(-50%);
z-index:9999;background:white;padding:8px 14px;border-radius:6px;
box-shadow:0 1px 4px rgba(0,0,0,0.3);text-align:center;
font-family:Arial;font-size:14px">
<b>Waste-bin observations, IIT Tirupati</b><br>
<span style="font-size:12px">Collected by Aditi Saha · QField · Sep 2026</span>
</div>
"""

legend_html = """
<div style="position:fixed;bottom:40px;left:12px;z-index:9999;
background:white;padding:8px 12px;border-radius:6px;
box-shadow:0 1px 4px rgba(0,0,0,0.3);font-family:Arial;font-size:12px">
<b>Legend</b><br>
<span style="display:inline-block;width:10px;height:10px;
border-radius:50%;background:#377eb8;margin-right:6px"></span>
Waste-bin observation
</div>
"""

m.get_root().html.add_child(Element(title_html))
m.get_root().html.add_child(Element(legend_html))

# Place the north arrow near the top-centre of the points
north_lat = dustbins.geometry.y.max()
north_lon = dustbins.geometry.x.mean()
add_north_arrow(m, north_lat, north_lon)

folium.LayerControl().add_to(m)

# Zoom so all points are visible
bounds = [
    [dustbins.geometry.y.min(), dustbins.geometry.x.min()],
    [dustbins.geometry.y.max(), dustbins.geometry.x.max()]
]
m.fit_bounds(bounds)

m


In [ ]:
# ------------------------------------------------------------
# 10. SAVE THE MAP AS HTML FOR THE WEBSITE
# ------------------------------------------------------------
m.save("web/waste-bins.html")
print("Saved web/waste-bins.html")
